In [1]:
# 2.1 Loading Necessary Libraries
import numpy as np  # linear algebra
import pandas as pd  # data processing
import matplotlib.pyplot as plt  # For basic data visualization.
import seaborn as sns  # For statistical data visualization
import warnings  # To manage warnings

# Suppressing FutureWarning
warnings.filterwarnings("ignore")

# 2.2 Loading Dataets
train_df = pd.read_csv("../data/spaceship-titanic/train.csv")
test_df = pd.read_csv("../data/spaceship-titanic/test.csv")

# Let's quickly check the data by viewing the first few rows.
print(train_df.head())
print(test_df.head())

  PassengerId HomePlanet CryoSleep  Cabin  Destination   Age    VIP  \
0     0001_01     Europa     False  B/0/P  TRAPPIST-1e  39.0  False   
1     0002_01      Earth     False  F/0/S  TRAPPIST-1e  24.0  False   
2     0003_01     Europa     False  A/0/S  TRAPPIST-1e  58.0   True   
3     0003_02     Europa     False  A/0/S  TRAPPIST-1e  33.0  False   
4     0004_01      Earth     False  F/1/S  TRAPPIST-1e  16.0  False   

   RoomService  FoodCourt  ShoppingMall     Spa  VRDeck               Name  \
0          0.0        0.0           0.0     0.0     0.0    Maham Ofracculy   
1        109.0        9.0          25.0   549.0    44.0       Juanna Vines   
2         43.0     3576.0           0.0  6715.0    49.0      Altark Susent   
3          0.0     1283.0         371.0  3329.0   193.0       Solam Susent   
4        303.0       70.0         151.0   565.0     2.0  Willy Santantines   

   Transported  
0        False  
1         True  
2        False  
3        False  
4         True  
  

In [24]:
y_train = train_df["Transported"]
train_len = len(train_df)

X_train_temp = train_df.drop(columns=["Transported"])

all_data = pd.concat([X_train_temp, test_df], axis=0).reset_index(drop=True)


def missing_data_table(df):
    total = (
        df.isnull().sum().sort_values(ascending=False)
    )  # Total missing values in each column.
    percent = (df.isnull().sum() / df.isnull().count()).sort_values(
        ascending=False
    )  # Percentage of missing values.

    missing_data = pd.concat(
        [total, percent], axis=1, keys=["Total", "Percent"]
    )  # Combine the results.

    # Filter columns that actually have missing values.
    return missing_data[missing_data["Total"] > 0]


# Display missing data for the training dataset.
missing_data = missing_data_table(all_data)
missing_data

,Total,Percent
CryoSleep,310,0.023901
ShoppingMall,306,0.023593
Cabin,299,0.023053
VIP,296,0.022822
Name,294,0.022668
FoodCourt,289,0.022282
HomePlanet,288,0.022205
Spa,284,0.021897
Destination,274,0.021126
Age,270,0.020817


In [25]:
all_data["CryoSleep"] = all_data["CryoSleep"].fillna("Unknown")
all_data["HomePlanet"] = all_data["HomePlanet"].fillna("Unknown")
all_data["Destination"] = all_data["Destination"].fillna("Unknown")

all_data["ShoppingMall"] = all_data["ShoppingMall"].fillna(
    all_data["ShoppingMall"].median()
)
all_data["FoodCourt"] = all_data["FoodCourt"].fillna(all_data["FoodCourt"].median())
all_data["Spa"] = all_data["Spa"].fillna(all_data["Spa"].median())
all_data["VRDeck"] = all_data["VRDeck"].fillna(all_data["VRDeck"].median())
all_data["RoomService"] = all_data["RoomService"].fillna(
    all_data["RoomService"].median()
)

all_data["total_spending"] = all_data[
    ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
].sum(axis=1)

all_data["Age"] = all_data["Age"].fillna(all_data["Age"].median())

all_data["CabinDeck"] = all_data["Cabin"].str.split("/").str[0]
all_data["CabinNum"] = all_data["Cabin"].str.split("/").str[1]
all_data["CabinSide"] = all_data["Cabin"].str.split("/").str[2]

all_data["CabinDeck"] = all_data["CabinDeck"].fillna("Unknown")
all_data["CabinNum"] = all_data["CabinNum"].fillna("Unknown")
all_data["CabinSide"] = all_data["CabinSide"].fillna("Unknown")

all_data["VIP"] = all_data["VIP"].apply(lambda x: 1 if x is True else 0)

all_data["last_name"] = all_data["Name"].str.split(" ").str[1]

all_data["last_name"] = all_data["last_name"].fillna("Unknown")

all_data = all_data.drop(columns=["Cabin", "Name"])

In [26]:
missing_data = missing_data_table(all_data)
missing_data

,Total,Percent


In [27]:
all_data["PassengerGroup"] = all_data["PassengerId"].str.split("_").str[0]

all_data["PassengerGroup_size"] = all_data.groupby("PassengerGroup")[
    "PassengerGroup"
].transform("count")


def categorize_group(size):
    if size == 1:
        return "Solo"
    elif size <= 4:
        return "Small_Group"
    else:
        return "Large_Group"


all_data["PassengerGroup_type"] = all_data["PassengerGroup_size"].apply(
    categorize_group
)

all_data["family_size"] = all_data.groupby(["PassengerGroup", "last_name"])[
    "PassengerId"
].transform("count")

In [28]:
missing_data = missing_data_table(all_data)
missing_data

,Total,Percent


In [29]:
all_data = all_data.drop(
    columns=[
        "PassengerId",
        "RoomService",
        "FoodCourt",
        "ShoppingMall",
        "Spa",
        "VRDeck",
        "PassengerGroup_size",
    ],
    errors="ignore",
)

X_train = all_data.iloc[:train_len, :]
X_test = all_data.iloc[train_len:, :]

In [30]:
from sklearn.model_selection import KFold, cross_val_score
import numpy as np


def log_loss_cv(model, X, y):
    kf = KFold(n_splits=10, shuffle=True, random_state=42)

    # REMOVE .values here! Pass the DataFrame 'X' directly.
    neg_log_losses = cross_val_score(model, X, y, scoring="neg_log_loss", cv=kf)

    return -neg_log_losses

In [31]:
X_train

,HomePlanet,CryoSleep,Destination,Age,VIP,total_spending,CabinDeck,CabinNum,CabinSide,last_name,PassengerGroup,PassengerGroup_type,family_size
0,Europa,False,TRAPPIST-1e,39.0,0,0.0,B,0,P,Ofracculy,0001,Solo,1
1,Earth,False,TRAPPIST-1e,24.0,0,736.0,F,0,S,Vines,0002,Solo,1
2,Europa,False,TRAPPIST-1e,58.0,1,10383.0,A,0,S,Susent,0003,Small_Group,2
3,Europa,False,TRAPPIST-1e,33.0,0,5176.0,A,0,S,Susent,0003,Small_Group,2
4,Earth,False,TRAPPIST-1e,16.0,0,1091.0,F,1,S,Santantines,0004,Solo,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
8688,Europa,False,55 Cancri e,41.0,1,8536.0,A,98,P,Noxnuther,9276,Solo,1
8689,Earth,True,PSO J318.5-22,18.0,0,0.0,G,1499,S,Mondalley,9278,Solo,1
8690,Earth,False,TRAPPIST-1e,26.0,0,1873.0,G,1500,S,Connon,9279,Solo,1
8691,Europa,False,55 Cancri e,32.0,0,4637.0,E,608,S,Hontichre,9280,Small_Group,2


In [32]:
missing_data = missing_data_table(X_train)
missing_data

,Total,Percent


In [33]:
from catboost import CatBoostClassifier

cat_feature_indices = [
    "HomePlanet",
    "CryoSleep",
    "Destination",
    "VIP",
    "CabinDeck",
    "CabinNum",
    "CabinSide",
    "last_name",
    "PassengerGroup",
    "PassengerGroup_type",
]

params = {
    # --- Core Configuration ---
    "loss_function": "Logloss",  # Standard for binary classification
    "eval_metric": "Accuracy",  # Useful to monitor during training
    "task_type": "GPU",  # Uses NVIDIA GPU for speed
    "random_seed": 42,  # For reproducibility
    # --- Training Speed & Depth ---
    "iterations": 2000,  # Number of trees
    "learning_rate": 0.02,  # Step size (lower = more robust)
    "depth": 6,  # Tree depth (6 is usually the 'sweet spot')
    "early_stopping_rounds": 50,  # Stop if no improvement
    # --- Overfitting Protection (Regularization) ---
    "l2_leaf_reg": 10,  # Penalty for leaf complexity
    "random_strength": 2,  # Randomness in split scoring
    "min_data_in_leaf": 15,  # Min samples per leaf
    "bagging_temperature": 1,  # Randomness of the bootstrap
    # --- Categorical Feature Handling ---
    "one_hot_max_size": 10,  # Categories with > 10 unique values get Target Encoded
    "model_size_reg": 0.5,  # Prevents high-cardinality features from bloating model size
    "max_ctr_complexity": 2,  # Limits the complexity of combining categorical features
    # --- GPU Specific Performance ---
    "gpu_ram_part": 0.9,  # Reserve 90% of GPU memory
    "bootstrap_type": "Bayesian",  # Required for bagging_temperature on GPU
    # --- Logging & Verbosity ---
    "verbose": 200,  # Print progress every 200 steps
    "allow_writing_files": False,  # Keeps your directory clean
}

model = CatBoostClassifier(**params, cat_features=cat_feature_indices)

catboost_rmse = log_loss_cv(model, X_train, y_train)

0:	learn: 0.7408922	total: 65.2ms	remaining: 2m 10s
200:	learn: 0.7527803	total: 10.3s	remaining: 1m 32s
400:	learn: 0.7585325	total: 20.3s	remaining: 1m 20s
600:	learn: 0.7636457	total: 29.9s	remaining: 1m 9s
800:	learn: 0.7676083	total: 39.1s	remaining: 58.5s
1000:	learn: 0.7705484	total: 48.8s	remaining: 48.7s
1200:	learn: 0.7736163	total: 58.3s	remaining: 38.8s
1400:	learn: 0.7764285	total: 1m 7s	remaining: 28.9s
1600:	learn: 0.7782181	total: 1m 17s	remaining: 19.3s
1800:	learn: 0.7796242	total: 1m 26s	remaining: 9.59s
1999:	learn: 0.7814138	total: 1m 37s	remaining: 0us
0:	learn: 0.7411479	total: 60.1ms	remaining: 2m
200:	learn: 0.7509907	total: 11.6s	remaining: 1m 44s
400:	learn: 0.7562316	total: 23.1s	remaining: 1m 32s
600:	learn: 0.7608334	total: 33.8s	remaining: 1m 18s
800:	learn: 0.7617282	total: 44s	remaining: 1m 5s
1000:	learn: 0.7647961	total: 54.4s	remaining: 54.3s
1200:	learn: 0.7672248	total: 1m 4s	remaining: 43s
1400:	learn: 0.7690144	total: 1m 14s	remaining: 32s
1600:	

In [ ]:
print(f"CatBoost RMSE (Cross-Validation Mean): {catboost_rmse.mean():.4f}") 
print(f"CatBoost RMSE (Standard Deviation): {catboost_rmse.std():.4f}") 

CatBoost RMSE (Cross-Validation Mean): 0.4960
CatBoost RMSE (Standard Deviation): 0.0157


In [35]:
model = CatBoostClassifier(**params, cat_features=cat_feature_indices)

model.fit(X_train, y_train)

0:	learn: 0.7400207	total: 45.6ms	remaining: 1m 31s
200:	learn: 0.7477281	total: 10.7s	remaining: 1m 35s
400:	learn: 0.7560106	total: 20.4s	remaining: 1m 21s
600:	learn: 0.7619924	total: 30.5s	remaining: 1m 11s
800:	learn: 0.7641781	total: 40.5s	remaining: 1m
1000:	learn: 0.7661337	total: 49.8s	remaining: 49.7s
1200:	learn: 0.7683193	total: 59.7s	remaining: 39.7s
1400:	learn: 0.7695847	total: 1m 9s	remaining: 29.7s
1600:	learn: 0.7711952	total: 1m 19s	remaining: 19.7s
1800:	learn: 0.7728057	total: 1m 28s	remaining: 9.8s
1999:	learn: 0.7740711	total: 1m 37s	remaining: 0us


In [37]:
id_col = test_df["PassengerId"]

def export_submission(model, id_col):
    preds = model.predict(X_test)

    submission = pd.DataFrame({"PassengerId": id_col, "Transported": preds})

    model_name = type(model).__name__
    submission.to_csv(f"{model_name}_submission.csv", index=False)


export_submission(model, id_col)